In [15]:
import os
import stat
import pathlib
import natsort

In [280]:
rootroot = pathlib.Path('dirwalker')

In [281]:
stat.S_ISDIR(os.stat(rootroot).st_mode)

True

In [282]:
natsort_key = natsort.natsort_keygen()

In [326]:
continuation = pathlib.Path('1/hello.txt').parts

In [421]:
def nextpath(continuation, n):
    # Find file entries that come after the continuation token in the chosen sort order.
    # If the continuation token is a directory, it descends into that directory.
    # It will list files until it encounters a directory so that the filtering
    # and sorting does not have to be repeated for folders containing many files.
    # The last entry of the result can be used as a continuation token to continue.
    # If there's nothing more to find, an empty list is returned.
    def result(base, snippets):
        return [ base / snippet for snippet in snippets[:n]]
            
    # we descend from the depth of the continuation token up
    # until we can match it with existing directories
    # and find sth to return
    for i in range(len(continuation)):
        subpath = pathlib.Path(*continuation[:len(continuation)-i])
        # The continuation token was an existing directory and our
        # next task is to descend into that directory for things.
        # No need to filter since the directory content comes after the containing
        # directory in the sort order
        if (rootroot / subpath).is_dir() and i == 0:
            target_dir = rootroot / subpath
            contents = sorted((x for x in os.listdir(target_dir) if x[0] != '.'), key=natsort_key)
            if contents:
                result = []
                base = subpath
                while contents and len(result) < n:
                    sn = contents.pop(0)
                    result.append(base / sn)
                    if (rootroot / base / sn).is_dir():
                        break
                return result
        # It is a file or doesn't exist
        else:
            # Look at contents one level up
            containing_dir = (rootroot / subpath).parent
            # We list the containing dir
            if containing_dir.is_dir():
                contents = sorted((x for x in os.listdir(containing_dir) if x[0] != '.'), key=natsort_key)
                # We filter out everything smaller or equal to the matching level of our continuation token
                key = natsort_key(continuation[len(continuation)-i-1])
                while contents and natsort_key(contents[0]) <= key:
                    contents.pop(0)
                if contents:
                    result = []
                    base = subpath.parent
                    while contents and len(result) < n:
                        sn = contents.pop(0)
                        result.append(base / sn)
                        # We found a diretory and need to descend now
                        # delegated to outer function
                        if (rootroot / base / sn).is_dir():
                            break
                    return result
                # We have reached the end
                elif len(subpath.parts) <= 0:
                    return []
    # Apparently we didn't find anything
    return []

def nextpath_outer(continuation, n):
    # nextpath() stops returning entries when it encounters a directory since it will need to descend.
    # That's why this outer function keeps calling nextpath() until our bucket is full
    result = []
    while len(result) < n:
        pieces = nextpath(continuation, n=n-len(result))
        if len(pieces) == 0:
            break
        result.extend(pieces)
        continuation = pathlib.Path(pieces[-1]).parts
    return result



In [419]:
continuation = pathlib.Path('2/c.txt').parts

while True:
    pieces = nextpath_outer(continuation, 1)
    print(pieces)
    if len(pieces) == 0:
        break
    continuation = pathlib.Path(pieces[-1]).parts

[PosixPath('2/d.txt')]
[PosixPath('a.txt')]
[PosixPath('b.txt')]
[PosixPath('d.txt')]
[]


In [354]:
pathlib.Path('asdf').is_dir()

False

In [396]:
[1, 2, 3].pop(0)

1